# コーパスの活用

大きなデータセットを活用した言語モデルの学習を学ぶ。


---

## コーパス

コーパス（corpus）またはテキストコーパスとは、言語学や自然言語処理における大規模なテキストデータの集合を指す。言語モデルの学習にもよく使用される。

hugging faceの[datasets](https://huggingface.co/docs/datasets/index)ライブラリを使うと、様々なコーパスを簡単に取得できる。[日本語wikipediaのコーパス](https://huggingface.co/datasets/llm-book/japanese-wikipedia)を取得してみよう。

In [1]:
import os; os.environ["HF_HUB_DISABLE_XET"] = "1" # なんかこれないとうまくダウンロードできない
from datasets import load_dataset

ds = load_dataset("llm-book/japanese-wikipedia")
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'meta'],
        num_rows: 1363395
    })
})

中身はこんな感じ。

In [2]:
# いくつかのサンプルの冒頭100文字を表示してみる
for i in range(3):
    print(ds["train"][i]["text"][:100])
    print("===")

『勝つか死ぬか』はHBO(日本ではスター・チャンネルが放送)のファンタジー・ドラマ・シリーズである『ゲーム・オブ・スローンズ』の第1章『七王国戦記』の第7話である。プロデューサーでもあるデイヴィッド・
===
ゲオルク（ヨーラン）・ヴァーレンベリ（Georg (Göran) Wahlenberg、1780年10月1日 – 1851年3月22日）は、スウェーデンの博物学者である。カール・ツンベルク（トゥーンベ
===
『進軍』はHBO(日本ではスター・チャンネルが放送)のファンタジー・ドラマ・シリーズである『ゲーム・オブ・スローンズ』の第1章『七王国戦記』の第8話である。原作小説シリーズ『氷と炎の歌』の作者で脚本家
===


こんなのが100万件以上入っている。

本来、webから取得したデータは非常に汚い。汚いというのは、HTMLタグや広告、重複データなどが含まれているということ。このようなテキストデータをそのまま学習に使用すると、当然モデルの性能は低下するため、前処理によって整える必要がある。しかしその作業は非常に手間がかかるため、本書では前処理済みのデータセットを使用させてもらう。


---

## トークナイザ

Tokenizer

テキストをトークンごとに分割するもの。言語モデルにおいて、モデルが扱う最小単位のことをトークンと呼び、単語や文字、サブワードなどが該当する。

全章では手動で文章を単語ごとに分割していたが、コーパスのような大規模なデータセットを扱う場合、手動でのトークン化は非現実的であるため、自動で分割するシステムが必要である。

文字をトークンとする場合、トークン化は非常に簡単で、そのまま1文字単位で区切ればいい。単語をトークンとする場合、英語の場合はスペースで区切るだけでよく、また日本語の場合は適当な形態素解析器を使う。しかしこれらの手法では文章を効率的に分割できないことが多い。効率的とは、より少ない語彙数で多くの文章を表せること。適切な語彙を設定していない場合、同じ文章を表すのに多くのトークンが必要となり、モデルの学習効率が低下する。

そこで、適切な分割方法をデータセットから学習する手法を用いる。具体的には、データセット内で頻出する文字列を1つのトークン（語彙）としてまとめる。これを指定した語彙数に達するまで繰り返し、効率的な分割方法を獲得する。この場合のトークンはサブワードと呼ばれたりする。

サブワードには基本的に単語よりも細かい分割を行うため、未知語への対応力が高いという利点もある。何ならBPE（Byte Pair Encoding）では文字よりもさらに細かいバイト列での分割を行うため、未知語への対応力はさらに高くなる。

実際にChatGPTで使われているBPEのトークナイザを[こちら](https://platform.openai.com/tokenizer)から試すことができ、入力した文章がどのように分割されているのかを確かめることができる。「鬱」とか「薔薇」とか、日常生活ではあまり使わない漢字を入れてみると「?」のアイコンがいくつか表示される。これは各漢字がバイト列に分割され、1文字を複数のトークンで表現しているということになる。

余談。

言語モデルについて、はじめにこう定義した。

> 言語モデルとは、単語の並びに関する確率モデルである。

この定義に従うなら、サブワードのような単語でないものをトークンとするモデル（現在有名な、おそらく全てのLLM）は言語モデルとは呼べないのかもしれない。

ただまあ、言語モデルというのは単語に限らず自然**言語**処理全般で活用されるモデルなので、単語にこだわりを持つ必要はないんじゃないかな。

hugging faceの[transformers](https://huggingface.co/docs/transformers/index)ライブラリから`AutoTokenizer`を使うと、学習済みのトークナイザを簡単に使用できる。

東北大の日本語BERTのトークナイザを取得してみよう。

In [3]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("tohoku-nlp/bert-base-japanese")

こんな感じでトークン化できる。

In [4]:
tokenizer.tokenize("近年の大規模言語モデルの発展は凄まじいね")

['近年', 'の', '大', '規模', '言語', 'モデル', 'の', '発展', 'は', '凄', '##まじ', '##い', 'ね']

`##`は前のトークンと連結していることを示す特殊な記号。


---

## コーパスを活用したマルコフモデルの学習

実際にこれらのコーパスやトークナイザを活用して、マルコフモデルの学習を行ってみよう。

まずは学習データを作成する。先の日本語wikiコーパスを使う。全部は多いので少しだけ。

In [5]:
ds_mini = ds["train"][:1000]
text = [" ".join(tokenizer.tokenize(t)) for t in ds_mini["text"]]

# samples
for t in text[:3]:
    print(t[:100])
    print("===")

『 勝つ か 死ぬ か 』 は H ##BO ( 日本 で は スター ・ チャンネル が 放送 ) の ファンタジー ・ ドラマ ・ シリーズ で ある 『 ゲーム ・ オブ ##・ ##ス ##ロ
===
ゲオルク ( ヨー ラン ) ・ ヴァー ##レン ##ベリ ( Geor ##g ( G ##ö ##ran ) W ##ah ##le ##n ##berg 、 1780 年 10 月 1 日 –
===
『 進軍 』 は H ##BO ( 日本 で は スター ・ チャンネル が 放送 ) の ファンタジー ・ ドラマ ・ シリーズ で ある 『 ゲーム ・ オブ ##・ ##ス ##ローン ##ズ 
===


これを`markovify`で学習させる。前章でできなかったN=2を試してみる。

In [14]:
import markovify

model = markovify.Text(text, state_size=2)

文章を生成してみる。

In [16]:
for _ in range(3):
    sentence = model.make_sentence()
    print(tokenizer.convert_tokens_to_string(sentence.split()))
    print("===")

カガチ 。 機動 戦士 V ガンダム の 登場 人物 、 フォンセ・カガチ を 参照 。
===
グネーシン 音楽 大学 の 人物 一覧 は 、 中華 人民 共和 国 の 通常 自治 体 の 首長 。
===
コシンジュガヤ に 似 て 、 より 小さい 事 から 。 分布 と 生育 環境 が 減少 傾向 に あっ て 各地 で 注意 さ れ た ジブラルタル で 行わ れる サッカー の カップ 戦 で ある 。 概要 1950 年 創設 さ れ た 。 過去 の 成績
===


語彙が増えたことで多様な文章が生成されるようになった。